# Supplementary Evaluation for Summarice Product Pipelines

This notebook is a supplementary product-level evaluation for Summarice's AI-assisted academic reading workflows. It complements the primary pure TLDR/model-summary evaluation in `docs/evals/evals.ipynb`, which tests summarization capability with ROUGE and BERTScore. This notebook evaluates three app-integrated pipelines using current app data: **citation-backed summary generation**, **figure interpretation**, and **Deep library search**.

The evaluation method is **LLM-assisted rubric review with manual validation**. An LLM judge can draft scores using the rubrics below, but the team manually reviews the scored outputs before reporting final results.

Dataset source: current Summarice app data, including uploaded documents, user highlights, notes, generated summaries, area-highlight figure interpretations, and Deep library search results. This is not intended to replace the TLDR benchmark evaluation; it checks whether the integrated product behavior remains grounded, navigable, and useful.

Latency is reported as two separate values:

- **Time to first response**: time until the app shows a streamed response, status update, or first result. This is the primary responsiveness metric and should be under 2 seconds when possible.
- **Time to completion**: time until the full AI output or result set is finished.

## Metric Definitions

Scores use a 0-2 rubric unless otherwise stated:

- `2`: correct, grounded, and useful.
- `1`: partially correct or useful, but missing important detail or containing minor issues.
- `0`: incorrect, unsupported, malformed, or not useful.

### Summary generation

- **Grounded citation quality**: summary claims are supported by the cited highlights, comments, or notes.
- **Citation validity**: every citation uses a valid highlight ordinal from the generated summary context.
- **Structure validity**: output contains the expected structured fields: `markdown`, `tags`, `entities`, and `open_questions`.
- **Coverage**: summary captures the document's main problem, method, contribution, and important open questions when present.

### Figure interpretation

- **Visual faithfulness**: the note describes only what is visible in the area-highlight screenshot.
- **Unsupported-claim avoidance**: the model avoids claims that cannot be verified from the image.
- **Usefulness**: the interpretation helps the reader understand the chart, diagram, table, or figure.
- **Failure behavior**: unclear or non-figure screenshots receive the prescribed failure response.

### Deep library search

- **Hit@3**: a relevant document or highlight appears in the top 3 results. This is the primary retrieval metric.
- **Hit@1**: the first result is relevant.
- **Reason quality**: the short relevance reason matches the result evidence.
- **Navigation correctness**: the result opens the expected document or highlight destination.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

PASS_THRESHOLD = 1.5

summary_evals = pd.DataFrame([
    {
        "case_id": "summary-01",
        "document": "Replace with document title",
        "grounded_citation_quality": 2,
        "citation_validity": 2,
        "structure_validity": 2,
        "coverage": 1,
        "time_to_first_response_s": 1.2,
        "time_to_completion_s": 12.4,
        "notes": "Replace with observed result notes."
    },
    {
        "case_id": "summary-02",
        "document": "Replace with document title",
        "grounded_citation_quality": 1,
        "citation_validity": 2,
        "structure_validity": 2,
        "coverage": 1,
        "time_to_first_response_s": 1.5,
        "time_to_completion_s": 14.1,
        "notes": "Replace with observed result notes."
    },
    {
        "case_id": "summary-03",
        "document": "Replace with document title",
        "grounded_citation_quality": 2,
        "citation_validity": 2,
        "structure_validity": 2,
        "coverage": 2,
        "time_to_first_response_s": 1.0,
        "time_to_completion_s": 11.8,
        "notes": "Replace with observed result notes."
    },
    {
        "case_id": "summary-04",
        "document": "Replace with document title",
        "grounded_citation_quality": 1,
        "citation_validity": 2,
        "structure_validity": 2,
        "coverage": 1,
        "time_to_first_response_s": 1.7,
        "time_to_completion_s": 15.3,
        "notes": "Replace with observed result notes."
    },
    {
        "case_id": "summary-05",
        "document": "Replace with document title",
        "grounded_citation_quality": 2,
        "citation_validity": 2,
        "structure_validity": 2,
        "coverage": 2,
        "time_to_first_response_s": 1.4,
        "time_to_completion_s": 13.0,
        "notes": "Replace with observed result notes."
    }
])

figure_evals = pd.DataFrame([
    {
        "case_id": "figure-01",
        "document": "Replace with document title",
        "highlight_description": "Replace with figure or area highlight description",
        "visual_faithfulness": 2,
        "unsupported_claim_avoidance": 2,
        "usefulness": 2,
        "failure_behavior": 2,
        "time_to_first_response_s": 1.3,
        "time_to_completion_s": 8.2,
        "notes": "Replace with observed result notes."
    },
    {
        "case_id": "figure-02",
        "document": "Replace with document title",
        "highlight_description": "Replace with figure or area highlight description",
        "visual_faithfulness": 2,
        "unsupported_claim_avoidance": 1,
        "usefulness": 2,
        "failure_behavior": 2,
        "time_to_first_response_s": 1.6,
        "time_to_completion_s": 9.5,
        "notes": "Replace with observed result notes."
    },
    {
        "case_id": "figure-03",
        "document": "Replace with document title",
        "highlight_description": "Replace with figure or area highlight description",
        "visual_faithfulness": 1,
        "unsupported_claim_avoidance": 2,
        "usefulness": 1,
        "failure_behavior": 2,
        "time_to_first_response_s": 1.4,
        "time_to_completion_s": 7.8,
        "notes": "Replace with observed result notes."
    },
    {
        "case_id": "figure-04",
        "document": "Replace with document title",
        "highlight_description": "Replace with figure or area highlight description",
        "visual_faithfulness": 2,
        "unsupported_claim_avoidance": 2,
        "usefulness": 2,
        "failure_behavior": 2,
        "time_to_first_response_s": 1.1,
        "time_to_completion_s": 7.4,
        "notes": "Replace with observed result notes."
    },
    {
        "case_id": "figure-05",
        "document": "Replace with document title",
        "highlight_description": "Replace with figure or area highlight description",
        "visual_faithfulness": 2,
        "unsupported_claim_avoidance": 2,
        "usefulness": 1,
        "failure_behavior": 2,
        "time_to_first_response_s": 1.8,
        "time_to_completion_s": 10.2,
        "notes": "Replace with observed result notes."
    }
])

search_evals = pd.DataFrame([
    {
        "case_id": f"search-{index:02d}",
        "query": "Replace with Deep library search query",
        "expected_result": "Replace with expected document or highlight",
        "hit_at_1": int(index in {1, 2, 4, 5, 7, 9}),
        "hit_at_3": int(index in {1, 2, 3, 4, 5, 6, 7, 9, 10}),
        "reason_quality": 2 if index in {1, 2, 4, 7, 9} else 1,
        "navigation_correctness": 2,
        "time_to_first_response_s": [0.8, 1.0, 1.1, 0.9, 1.3, 1.4, 1.2, 1.6, 0.7, 1.5][index - 1],
        "time_to_completion_s": [2.4, 2.9, 3.1, 2.6, 3.5, 3.8, 3.2, 4.0, 2.2, 3.6][index - 1],
        "notes": "Replace with observed result notes."
    }
    for index in range(1, 11)
])

summary_evals, figure_evals, search_evals

In [ ]:
import json
import os
from pathlib import Path
from urllib.parse import quote, urlencode
from urllib.request import Request, urlopen


def load_dotenv_file(path):
    env_path = Path(path)
    if not env_path.exists():
        return

    for raw_line in env_path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


def candidate_env_paths():
    roots = [Path.cwd(), Path.cwd() / "docs" / "evals"]
    paths = []
    for root in roots:
        for current in [root, *root.parents]:
            paths.extend([current / ".env.local", current / ".env"])
    seen = set()
    return [path for path in paths if not (str(path) in seen or seen.add(str(path)))]


for env_file in candidate_env_paths():
    load_dotenv_file(env_file)

SUPABASE_URL = os.environ.get("PUBLIC_SUPABASE_URL", "").rstrip("/")
SUPABASE_SERVICE_ROLE_KEY = os.environ.get("SUPABASE_SERVICE_ROLE_KEY")
SUPABASE_ANON_KEY = os.environ.get("PUBLIC_SUPABASE_ANON_KEY")
SUPABASE_KEY = SUPABASE_SERVICE_ROLE_KEY or SUPABASE_ANON_KEY


def fetch_supabase_rows(table, params):
    if not SUPABASE_URL or not SUPABASE_KEY:
        return []

    query = urlencode(params, doseq=True, safe=",():.*")
    request = Request(
        f"{SUPABASE_URL}/rest/v1/{quote(table)}?{query}",
        headers={
            "apikey": SUPABASE_KEY,
            "Authorization": f"Bearer {SUPABASE_KEY}",
            "Accept": "application/json"
        }
    )

    with urlopen(request, timeout=20) as response:
        return json.loads(response.read().decode("utf-8"))


def nested_document_title(row):
    document = row.get("documents") or {}
    if isinstance(document, dict):
        return document.get("title") or "Untitled document"
    return "Untitled document"


def nested_highlight_document_title(row):
    highlight = row.get("highlights") or {}
    if not isinstance(highlight, dict):
        return "Untitled document"
    document = highlight.get("documents") or {}
    if isinstance(document, dict):
        return document.get("title") or "Untitled document"
    return "Untitled document"


def score_placeholder(note):
    return {
        "time_to_first_response_s": pd.NA,
        "time_to_completion_s": pd.NA,
        "notes": note
    }


def build_summary_rows(rows):
    return pd.DataFrame([
        {
            "case_id": f"summary-{index:02d}",
            "summary_id": row.get("id"),
            "created_at": row.get("created_at"),
            "document": nested_document_title(row),
            "grounded_citation_quality": pd.NA,
            "citation_validity": pd.NA,
            "structure_validity": pd.NA,
            "coverage": pd.NA,
            **score_placeholder("Score this generated summary with the rubric.")
        }
        for index, row in enumerate(rows[:5], start=1)
    ])


def build_figure_rows(rows):
    return pd.DataFrame([
        {
            "case_id": f"figure-{index:02d}",
            "annotation_id": row.get("id"),
            "created_at": row.get("created_at"),
            "document": nested_highlight_document_title(row),
            "highlight_description": f"Area highlight AI note created at {row.get('created_at')}",
            "visual_faithfulness": pd.NA,
            "unsupported_claim_avoidance": pd.NA,
            "usefulness": pd.NA,
            "failure_behavior": pd.NA,
            **score_placeholder("Score this figure interpretation with the rubric.")
        }
        for index, row in enumerate(rows[:5], start=1)
    ])


def search_completion_seconds(row):
    telemetry = row.get("telemetry")
    if isinstance(telemetry, str):
        try:
            telemetry = json.loads(telemetry)
        except json.JSONDecodeError:
            telemetry = {}
    if isinstance(telemetry, dict) and telemetry.get("latencyMs") is not None:
        return telemetry["latencyMs"] / 1000
    if row.get("latency_ms") is not None:
        return row["latency_ms"] / 1000
    return pd.NA


def build_search_rows(rows):
    return pd.DataFrame([
        {
            "case_id": f"search-{index:02d}",
            "search_id": row.get("id"),
            "created_at": row.get("created_at"),
            "query": row.get("raw_query") or row.get("query") or row.get("text_query") or "",
            "expected_result": "Fill with relevant expected document or highlight after review.",
            "hit_at_1": pd.NA,
            "hit_at_3": pd.NA,
            "reason_quality": pd.NA,
            "navigation_correctness": pd.NA,
            "time_to_first_response_s": pd.NA,
            "time_to_completion_s": search_completion_seconds(row),
            "notes": "Run this query in Deep library search and score the top results."
        }
        for index, row in enumerate(rows[:10], start=1)
    ])


try:
    real_summary_rows = fetch_supabase_rows(
        "summaries",
        {
            "select": "id,document_id,created_at,is_current,markdown,tags,entities,open_questions,documents(title)",
            "is_current": "eq.true",
            "order": "created_at.desc",
            "limit": "5"
        }
    )
    real_figure_rows = fetch_supabase_rows(
        "annotations",
        {
            "select": "id,highlight_id,created_at,body,highlights(id,page_number,kind,documents(title))",
            "source": "eq.ai",
            "order": "created_at.desc",
            "limit": "5"
        }
    )
    real_search_rows = fetch_supabase_rows(
        "searches",
        {
            "select": "id,query,text_query,latency_ms,created_at",
            "order": "created_at.desc",
            "limit": "10"
        }
    )

    if len(real_summary_rows) >= 1:
        summary_evals = build_summary_rows(real_summary_rows)
    if len(real_figure_rows) >= 1:
        figure_evals = build_figure_rows(real_figure_rows)
    if len(real_search_rows) >= 1:
        search_evals = build_search_rows(real_search_rows)

    print(
        "Loaded most recent rows by created_at descending:",
        f"{len(real_summary_rows)} current summaries,",
        f"{len(real_figure_rows)} figure interpretations,",
        f"{len(real_search_rows)} searches."
    )
    if not SUPABASE_SERVICE_ROLE_KEY:
        print("Note: SUPABASE_SERVICE_ROLE_KEY is missing. The anon key can return zero rows because Supabase RLS requires an authenticated user.")
    if len(real_summary_rows) == 0 and len(real_figure_rows) == 0 and len(real_search_rows) == 0:
        print("No real examples were loaded. Add SUPABASE_SERVICE_ROLE_KEY to .env.local or fill the tables manually.")
except Exception as exc:
    print("Using editable placeholder rows because automatic Supabase loading failed.")
    print(type(exc).__name__, str(exc))


display(summary_evals)
display(figure_evals)
display(search_evals)

In [ ]:
final_judged_scores = [
    {"case_id": "summary-01", "pipeline": "summary_generation", "scores": {"grounded_citation_quality": 1, "citation_validity": 2, "structure_validity": 2, "coverage": 1}, "recommended_notes": "Valid structure and citations, but several benchmark and implementation details need stronger cited evidence."},
    {"case_id": "summary-02", "pipeline": "summary_generation", "scores": {"grounded_citation_quality": 1, "citation_validity": 2, "structure_validity": 2, "coverage": 2}, "recommended_notes": "Strong coverage, but tighten citations around coordination and straggler claims."},
    {"case_id": "summary-03", "pipeline": "summary_generation", "scores": {"grounded_citation_quality": 1, "citation_validity": 2, "structure_validity": 2, "coverage": 2}, "recommended_notes": "Good structure and coverage; some technical/result claims need more direct cited text evidence."},
    {"case_id": "summary-04", "pipeline": "summary_generation", "scores": {"grounded_citation_quality": 1, "citation_validity": 2, "structure_validity": 2, "coverage": 1}, "recommended_notes": "Acceptable minimal summary, but coverage is thin and one claim is weakly supported."},
    {"case_id": "summary-05", "pipeline": "summary_generation", "scores": {"grounded_citation_quality": 2, "citation_validity": 2, "structure_validity": 2, "coverage": 2}, "recommended_notes": "Well-grounded summary with valid citations and good coverage of the selected material."},
    {"case_id": "figure-01", "pipeline": "figure_interpretation", "scores": {"visual_faithfulness": 2, "unsupported_claim_avoidance": 2, "usefulness": 2, "failure_behavior": 2}, "recommended_notes": "Faithful and useful description of the process diagram."},
    {"case_id": "figure-02", "pipeline": "figure_interpretation", "scores": {"visual_faithfulness": 2, "unsupported_claim_avoidance": 2, "usefulness": 2, "failure_behavior": 2}, "recommended_notes": "Accurate, useful interpretation of the multi-panel figure."},
    {"case_id": "figure-03", "pipeline": "figure_interpretation", "scores": {"visual_faithfulness": 2, "unsupported_claim_avoidance": 2, "usefulness": 2, "failure_behavior": 2}, "recommended_notes": "Faithful explanation of the scaling plots and their takeaway."},
    {"case_id": "figure-04", "pipeline": "figure_interpretation", "scores": {"visual_faithfulness": 2, "unsupported_claim_avoidance": 2, "usefulness": 2, "failure_behavior": 2}, "recommended_notes": "Good high-level map of the diagram without hallucinated details."},
    {"case_id": "figure-05", "pipeline": "figure_interpretation", "scores": {"visual_faithfulness": 2, "unsupported_claim_avoidance": 2, "usefulness": 2, "failure_behavior": 2}, "recommended_notes": "Faithful and helpful interpretation of the embedding visualization."},
    {"case_id": "search-01", "pipeline": "deep_library_search", "scores": {"hit_at_1": 1, "hit_at_3": 1, "reason_quality": 2, "navigation_correctness": 2}, "recommended_notes": "Manual review marked this search accurate; the relevant result was retrieved and navigable."},
    {"case_id": "search-02", "pipeline": "deep_library_search", "scores": {"hit_at_1": 1, "hit_at_3": 1, "reason_quality": 2, "navigation_correctness": 2}, "recommended_notes": "Manual review marked this search accurate; the relevant result was retrieved and navigable."},
    {"case_id": "search-03", "pipeline": "deep_library_search", "scores": {"hit_at_1": 1, "hit_at_3": 1, "reason_quality": 2, "navigation_correctness": 2}, "recommended_notes": "Manual review marked this search accurate; the relevant result was retrieved and navigable."},
    {"case_id": "search-04", "pipeline": "deep_library_search", "scores": {"hit_at_1": 1, "hit_at_3": 1, "reason_quality": 2, "navigation_correctness": 2}, "recommended_notes": "Manual review marked this search accurate; the relevant result was retrieved and navigable."},
    {"case_id": "search-05", "pipeline": "deep_library_search", "scores": {"hit_at_1": 1, "hit_at_3": 1, "reason_quality": 2, "navigation_correctness": 2}, "recommended_notes": "Manual review marked this search accurate; the relevant result was retrieved and navigable."},
    {"case_id": "search-06", "pipeline": "deep_library_search", "scores": {"hit_at_1": 1, "hit_at_3": 1, "reason_quality": 2, "navigation_correctness": 2}, "recommended_notes": "Manual review marked this search accurate; the relevant result was retrieved and navigable."},
    {"case_id": "search-07", "pipeline": "deep_library_search", "scores": {"hit_at_1": 1, "hit_at_3": 1, "reason_quality": 2, "navigation_correctness": 2}, "recommended_notes": "Manual review marked this search accurate; the relevant result was retrieved and navigable."},
    {"case_id": "search-08", "pipeline": "deep_library_search", "scores": {"hit_at_1": 0, "hit_at_3": 0, "reason_quality": 0, "navigation_correctness": 0}, "recommended_notes": "Manual review marked this as the one inaccurate Deep library search case."},
    {"case_id": "search-09", "pipeline": "deep_library_search", "scores": {"hit_at_1": 1, "hit_at_3": 1, "reason_quality": 2, "navigation_correctness": 2}, "recommended_notes": "Manual review marked this search accurate; the relevant result was retrieved and navigable."},
    {"case_id": "search-10", "pipeline": "deep_library_search", "scores": {"hit_at_1": 1, "hit_at_3": 1, "reason_quality": 2, "navigation_correctness": 2}, "recommended_notes": "Manual review marked this search accurate; the relevant result was retrieved and navigable."}
]


def apply_judged_scores(frame, pipeline):
    next_frame = frame.copy()
    score_rows = [row for row in final_judged_scores if row["pipeline"] == pipeline]
    for score_row in score_rows:
        row_mask = next_frame["case_id"] == score_row["case_id"]
        for metric, value in score_row["scores"].items():
            if metric in next_frame.columns:
                next_frame.loc[row_mask, metric] = value
        next_frame.loc[row_mask, "notes"] = score_row["recommended_notes"]
    return next_frame


summary_evals = apply_judged_scores(summary_evals, "summary_generation")
figure_evals = apply_judged_scores(figure_evals, "figure_interpretation")
search_evals = apply_judged_scores(search_evals, "deep_library_search")

print("Applied final judged scores:", len(final_judged_scores), "cases")
display(summary_evals)
display(figure_evals)
display(search_evals)

In [ ]:
def add_average_score(frame, score_columns):
    next_frame = frame.copy()
    numeric_scores = next_frame[score_columns].apply(pd.to_numeric, errors="coerce")
    next_frame["scored_metric_count"] = numeric_scores.notna().sum(axis=1)
    next_frame["average_score"] = numeric_scores.mean(axis=1)
    next_frame["passed"] = next_frame["average_score"].ge(PASS_THRESHOLD).where(
        next_frame["average_score"].notna(),
        pd.NA
    )
    return next_frame


def reviewed_pass_rate(frame):
    reviewed = frame["passed"].dropna()
    if reviewed.empty:
        return pd.NA
    return reviewed.astype(bool).mean()


def reviewed_average(frame, column):
    values = pd.to_numeric(frame[column], errors="coerce").dropna()
    if values.empty:
        return pd.NA
    return values.mean()


def reviewed_count(frame):
    return int(frame["average_score"].notna().sum())


summary_scored = add_average_score(
    summary_evals,
    ["grounded_citation_quality", "citation_validity", "structure_validity", "coverage"]
)

figure_scored = add_average_score(
    figure_evals,
    ["visual_faithfulness", "unsupported_claim_avoidance", "usefulness", "failure_behavior"]
)

search_scored = add_average_score(
    search_evals,
    ["hit_at_3", "reason_quality", "navigation_correctness"]
)

pipeline_summary = pd.DataFrame([
    {
        "pipeline": "Summary generation",
        "examples_loaded": len(summary_scored),
        "examples_scored": reviewed_count(summary_scored),
        "average_score": reviewed_average(summary_scored, "average_score"),
        "pass_rate": reviewed_pass_rate(summary_scored),
        "average_time_to_first_response_s": reviewed_average(summary_scored, "time_to_first_response_s"),
        "average_time_to_completion_s": reviewed_average(summary_scored, "time_to_completion_s")
    },
    {
        "pipeline": "Figure interpretation",
        "examples_loaded": len(figure_scored),
        "examples_scored": reviewed_count(figure_scored),
        "average_score": reviewed_average(figure_scored, "average_score"),
        "pass_rate": reviewed_pass_rate(figure_scored),
        "average_time_to_first_response_s": reviewed_average(figure_scored, "time_to_first_response_s"),
        "average_time_to_completion_s": reviewed_average(figure_scored, "time_to_completion_s")
    },
    {
        "pipeline": "Deep library search",
        "examples_loaded": len(search_scored),
        "examples_scored": reviewed_count(search_scored),
        "average_score": reviewed_average(search_scored, "average_score"),
        "pass_rate": reviewed_pass_rate(search_scored),
        "average_time_to_first_response_s": reviewed_average(search_scored, "time_to_first_response_s"),
        "average_time_to_completion_s": reviewed_average(search_scored, "time_to_completion_s")
    }
])

search_metrics = pd.DataFrame([
    {"metric": "Hit@1", "value": reviewed_average(search_scored, "hit_at_1")},
    {"metric": "Hit@3", "value": reviewed_average(search_scored, "hit_at_3")}
])

if pipeline_summary["examples_scored"].sum() == 0:
    print("Real examples are loaded, but rubric scores are still blank. Fill the score columns, then rerun this cell and the charts.")

if search_metrics["value"].isna().all():
    print("Search Hit@1/Hit@3 are blank until you score the loaded Deep library search rows.")

display(pipeline_summary.round(3))
display(search_metrics.round(3))

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

from pathlib import Path

def resolve_eval_figure_dir():
    for root in [Path.cwd(), *Path.cwd().parents]:
        candidate = root / "docs" / "documentation" / "assets" / "evals"
        if (root / "docs" / "documentation").exists():
            candidate.mkdir(parents=True, exist_ok=True)
            return candidate
    fallback = Path("assets") / "evals"
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback


EVAL_FIGURE_DIR = resolve_eval_figure_dir()


def plot_bar_or_message(axis, frame, x_column, y_column, title, ylabel, color, ylim=None, message="Not scored yet"):
    values = pd.to_numeric(frame[y_column], errors="coerce")
    plot_frame = frame.copy()
    plot_frame[y_column] = values

    if values.notna().any():
        plot_frame.plot(
            x=x_column,
            y=y_column,
            kind="bar",
            ax=axis,
            legend=False,
            color=color,
            ylim=ylim
        )
        axis.set_xlabel("")
        axis.set_ylabel(ylabel)
        return

    axis.text(
        0.5,
        0.5,
        message,
        ha="center",
        va="center",
        fontsize=12,
        color="#525252",
        transform=axis.transAxes
    )
    axis.set_xticks([])
    axis.set_yticks([])
    axis.set_xlabel("")
    axis.set_ylabel(ylabel)
    if ylim is not None:
        axis.set_ylim(*ylim)


def save_single_bar(frame, x_column, y_column, title, ylabel, color, filename, ylim=None, message="Not scored yet", add_latency_target=False):
    fig, axis = plt.subplots(figsize=(8, 5))
    plot_bar_or_message(axis, frame, x_column, y_column, title, ylabel, color, ylim=ylim, message=message)
    if add_latency_target:
        axis.axhline(2, color="#dc2626", linestyle="--", linewidth=1, label="2s target")
        axis.legend()
    axis.set_title(title)
    axis.tick_params(axis="x", rotation=20)
    fig.tight_layout()
    fig.savefig(EVAL_FIGURE_DIR / filename, dpi=200, bbox_inches="tight")
    return fig


fig, axes = plt.subplots(2, 2, figsize=(14, 9))

plot_bar_or_message(
    axes[0, 0],
    pipeline_summary,
    "pipeline",
    "average_score",
    "Average Rubric Score by Pipeline",
    "Average score (0-2)",
    "#2563eb",
    ylim=(0, 2),
    message="Fill rubric scores first"
)
axes[0, 0].set_title("Average Rubric Score by Pipeline")

plot_bar_or_message(
    axes[0, 1],
    pipeline_summary,
    "pipeline",
    "pass_rate",
    "Pass Rate by Pipeline",
    "Pass rate",
    "#16a34a",
    ylim=(0, 1),
    message="Fill rubric scores first"
)
axes[0, 1].set_title("Pass Rate by Pipeline")

plot_bar_or_message(
    axes[1, 0],
    pipeline_summary,
    "pipeline",
    "average_time_to_first_response_s",
    "Average Time to First Response",
    "Seconds",
    "#f97316",
    message="Add first-response timings"
)
axes[1, 0].axhline(2, color="#dc2626", linestyle="--", linewidth=1, label="2s target")
axes[1, 0].legend()
axes[1, 0].set_title("Average Time to First Response")

plot_bar_or_message(
    axes[1, 1],
    search_metrics,
    "metric",
    "value",
    "Deep Library Search Hit Rate",
    "Hit rate",
    "#7c3aed",
    ylim=(0, 1),
    message="Score Hit@1 and Hit@3 first"
)
axes[1, 1].set_title("Deep Library Search Hit Rate")

for axis in axes.flat:
    axis.tick_params(axis="x", rotation=20)

plt.tight_layout()
fig.savefig(EVAL_FIGURE_DIR / "evaluation_dashboard.png", dpi=200, bbox_inches="tight")

save_single_bar(
    pipeline_summary,
    "pipeline",
    "average_score",
    "Average Rubric Score by Pipeline",
    "Average score (0-2)",
    "#2563eb",
    "average_rubric_score.png",
    ylim=(0, 2),
    message="Fill rubric scores first"
)
save_single_bar(
    pipeline_summary,
    "pipeline",
    "pass_rate",
    "Pass Rate by Pipeline",
    "Pass rate",
    "#16a34a",
    "pass_rate_by_pipeline.png",
    ylim=(0, 1),
    message="Fill rubric scores first"
)
save_single_bar(
    pipeline_summary,
    "pipeline",
    "average_time_to_first_response_s",
    "Average Time to First Response",
    "Seconds",
    "#f97316",
    "time_to_first_response.png",
    message="Add first-response timings",
    add_latency_target=True
)
save_single_bar(
    search_metrics,
    "metric",
    "value",
    "Deep Library Search Hit Rate",
    "Hit rate",
    "#7c3aed",
    "deep_search_hit_rate.png",
    ylim=(0, 1),
    message="Score Hit@1 and Hit@3 first"
)

print("Exported evaluation figures to", EVAL_FIGURE_DIR.resolve())
plt.show()

## Discussion

This supplementary evaluation uses LLM-specific product metrics because these app-integrated features are not trained classifiers with labels, epochs, or prediction classes. The separate TLDR/model-summary evaluation remains the appropriate place for ROUGE and BERTScore results. Classic artifacts such as confusion matrices, ROC curves, and loss-vs-epoch plots are therefore not the best fit for this system. Instead, the evaluation focuses on whether each pipeline produces useful, grounded, and navigable outputs in the actual reading workflow.

For **summary generation**, the most important result is grounded citation quality. The summary pipeline is expected to produce structured summaries with citations that point back to valid highlights. This makes citation validity and groundedness more meaningful than generic text similarity alone.

For **figure interpretation**, the core risk is visual hallucination. The evaluation therefore prioritizes visual faithfulness and unsupported-claim avoidance. A strong result is not merely fluent; it must stay within what is visible in the selected area highlight.

For **Deep library search**, the output is a ranked set of navigable documents and highlights, not a generated chat answer. The primary metric is Hit@3 because the user experience depends on finding a relevant result quickly within the top few results. Hit@1 is also reported as a stricter secondary metric.

Latency is interpreted as responsiveness rather than full generation time. Since Summarice streams status updates and AI output, **time to first response** is the main requirement for perceived speed, while **time to completion** is reported separately for transparency.

Limitations: the evaluation uses a small set of current app examples, so the results should be interpreted as supplementary final-project validation rather than a broad benchmark or a replacement for the pure TLDR summary evaluation. Future work could add a larger offline evaluation harness, compare multiple model providers, and run repeated trials to estimate score variance.